# DistilBERT Base Model
The following contains the code to create and train a DistilBERT model using the Huggingface library. It works quite well for a moderate amount of data, but the runtime increases quite drastically with data.

I decided to take the pretrained model after all, still, creating the model myself was quite interesting!

In [1]:
from pathlib import Path
import torch
import time
from pathlib import Path
import os
from tqdm.auto import tqdm
from torch.optim import AdamW
import torchtest
from transformers import pipeline
from transformers import ElectraTokenizerFast
from transformers import ElectraConfig
from transformers import ElectraForMaskedLM

from electra import Dataset
from electra import test_model

import numpy as np

## Tokeniser
We need a way to convert the strings we get as the input to numerical tokens, that we can give to the neual network. Hence, we take a BertWorkPieceTokenizer (works for DistilBERT too) and create tokens from our words.

In [2]:
fit_new_tokenizer = False

if fit_new_tokenizer:
    paths = [str(x) for x in Path('babylm_data/babylm_10M').glob('**/*.txt')]

    tokenizer = ElectraTokenizerFast(
        clean_text=True,
        handle_chinese_chars=False,
        strip_accents=False,
        lowercase=True
    )

In [3]:
# fit the tokenizer
if fit_new_tokenizer:
    tokenizer.train(files=paths[:10], vocab_size=30_000, min_frequency=2,
                    limit_alphabet=1000, wordpieces_prefix='##',
                    special_tokens=['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]'])

In [4]:
if fit_new_tokenizer:
    os.mkdir('./tokeniser')
    tokenizer.save_model('tokeniser')

After having created a basic tokeniser, we use the model to initialise a DistilBert tokenizer, that we need for the model architecture later on. We save the tokeniser separately.

In [5]:
tokenizer = ElectraTokenizerFast.from_pretrained('google/electra-small-discriminator', max_len=512)
tokenizer.save_pretrained("electra_tokenizer")

('electra_tokenizer\\tokenizer_config.json',
 'electra_tokenizer\\tokenizer.json')

### Testing
We now test the created tokenizer. We take a simple example and tokenise the input. It can be seen that we add a special token in the beginning and end ('CLS' and 'SEP'), which is how the BERT model was defined.

When we translate the input back, we can see that we get the same, except for the first and last token. Also, we can see that questionmarks and commas are encoded separately.

In [6]:
tokens = tokenizer('Hello, how are you?')
print(tokens)

{'input_ids': [101, 7592, 1010, 2129, 2024, 2017, 1029, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}


In [7]:
tokenizer.decode(tokens['input_ids'])

'[CLS] hello, how are you? [SEP]'

In [8]:
for tok in tokens['input_ids']:
    print(tokenizer.decode(tok))

[CLS]
hello
,
how
are
you
?
[SEP]


In [9]:
assert len(tokenizer.vocab) == 30_522

## Dataset
We now define a function to mask some of the tokens. In particular, we create a Dataset class, that automates loading the data and tokenising it for us. Lastly, we use a DataLoader to load the data step by step into memory.

The big problem with the limited resources we have is memory. In particular, I am loading the data sequentially, file by file, keeping track how many samples have been read. Shuffling wouldn't work here (it would also not make a lot of sense for this dataset).

In [10]:
# dataset = Dataset(paths = [str(x) for x in Path('babylm_data/babylm_10M').glob('**/*.train')][0:20], tokenizer=tokenizer)
# loader = torch.utils.data.DataLoader(dataset, batch_size=8)

test_dataset = Dataset(paths = [str(x) for x in Path('babylm_data/babylm_test').glob('**/*.test')][10:12], tokenizer=tokenizer)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=4)

### Testing
The randomisation makes it a bit difficult to test. But altogether, we see that the input ids, masks and labels have the same shape. Also, as we mask 15% of the samples, when decoding a given sample, we can see that some samples are now '[MASK]'.

In [11]:
i = iter(test_dataset)

In [73]:
for j in range(10):
    sample = next(i)
    input_ids = sample['input_ids']
    attention_masks = sample['attention_mask']
    labels = sample['labels']
    
    # check if the dimensions are right
    assert input_ids.shape[0] == (512)
    assert attention_masks.shape[0] == (512)
    assert labels.shape[0] == (512)

    # if the input ids are not masked, the labels are the same as the input ids
    assert np.array_equal(input_ids[input_ids != 103].numpy(),labels[input_ids != 103].numpy())
    # input ids are zero if the attention masks are zero
    assert np.all(input_ids[attention_masks == 0].numpy() == 0)
    # check if input contains masked tokens (we can't guarantee this 100% but this will apply) most likely
    assert np.any(input_ids.numpy() == 103)
print("Passed")

Passed


## Model
In the following section, we intialise and train a model.

In [74]:
config = ElectraConfig(
    vocab_size=30000,
    max_position_embeddings=514
)

In [75]:
model = ElectraForMaskedLM(config)

In [76]:
# if we have a GPU - train on gpu
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

ElectraForMaskedLM(
  (electra): ElectraModel(
    (embeddings): ElectraEmbeddings(
      (word_embeddings): Embedding(30000, 128, padding_idx=0)
      (position_embeddings): Embedding(514, 128)
      (token_type_embeddings): Embedding(2, 128)
      (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (embeddings_project): Linear(in_features=128, out_features=256, bias=True)
    (encoder): ElectraEncoder(
      (layer): ModuleList(
        (0-11): 12 x ElectraLayer(
          (attention): ElectraAttention(
            (self): ElectraSelfAttention(
              (query): Linear(in_features=256, out_features=256, bias=True)
              (key): Linear(in_features=256, out_features=256, bias=True)
              (value): Linear(in_features=256, out_features=256, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): ElectraSelfOutput(
              (dense): Linear(in_featur

In [77]:
# get smaller dataset
test_ds = Dataset(paths = [str(x) for x in Path('babylm_data/babylm_test').glob('**/*.test')][:2], tokenizer=tokenizer)
test_ds_loader = torch.utils.data.DataLoader(test_ds, batch_size=2)
optim=torch.optim.Adam(model.parameters())

In [80]:
test_model(model, optim, test_ds_loader, device)

Passed


### Training the model
We use AdamW as the optimiser and train for 10 epochs.

Taking the whole dataset, takes about 100 hours per epoch for me, so I wasn't able to do that.

In [81]:
model = ElectraForMaskedLM(config)
# if we have a GPU - train on gpu
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

ElectraForMaskedLM(
  (electra): ElectraModel(
    (embeddings): ElectraEmbeddings(
      (word_embeddings): Embedding(30000, 128, padding_idx=0)
      (position_embeddings): Embedding(514, 128)
      (token_type_embeddings): Embedding(2, 128)
      (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (embeddings_project): Linear(in_features=128, out_features=256, bias=True)
    (encoder): ElectraEncoder(
      (layer): ModuleList(
        (0-11): 12 x ElectraLayer(
          (attention): ElectraAttention(
            (self): ElectraSelfAttention(
              (query): Linear(in_features=256, out_features=256, bias=True)
              (key): Linear(in_features=256, out_features=256, bias=True)
              (value): Linear(in_features=256, out_features=256, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): ElectraSelfOutput(
              (dense): Linear(in_featur

In [82]:
# we use AdamW as the optimiser
optim = AdamW(model.parameters(), lr=1e-4)

In [84]:
epochs = 10

for epoch in range(epochs):
    loop = tqdm(test_loader, leave=True)
    
    # set model to training mode
    model.train()
    losses = []
    
    # iterate over dataset
    for batch in loop:
        optim.zero_grad()
        
        # copy input to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # predict
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        
        # update weights
        loss = outputs.loss
        loss.backward()
        
        optim.step()
        
        # output current loss
        loop.set_description(f'Epoch {epoch}')
        loop.set_postfix(loss=loss.item())
        losses.append(loss.item())
        
        del input_ids
        del attention_mask
        del labels
        
    print("Mean Training Loss", np.mean(losses))
    losses = []
    loop = tqdm(test_loader, leave=True)
    
    # set model to evaluation mode
    model.eval()
    
    # iterate over dataset
    for batch in loop:
        # copy input to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # predict
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        
        # update weights
        loss = outputs.loss
        
        # output current loss
        loop.set_description(f'Epoch {epoch}')
        loop.set_postfix(loss=loss.item())
        losses.append(loss.item())
        
        del input_ids
        del attention_mask
        del labels
    print("Mean Test Loss", np.mean(losses))

  0%|          | 0/2500 [00:00<?, ?it/s]


KeyboardInterrupt



In [22]:
# save the pretrained model
torch.save(model, "electra.model")

In [25]:
model = torch.load("electra.model")

### Testing
Huggingface provides a library to quickly be able to see what word the model would predict for our masked token.

In [27]:
fill = pipeline("fill-mask", model='electra', config=config, tokenizer='electra_tokenizer')

In [28]:
fill(f'It seems important to tackle the climate {fill.tokenizer.mask_token}.')

[{'score': 0.19730663299560547,
  'token': 2965,
  'token_str': 'change',
  'sequence': 'it seems important to tackle the climate change.'},
 {'score': 0.12946806848049164,
  'token': 5215,
  'token_str': 'crisis',
  'sequence': 'it seems important to tackle the climate crisis.'},
 {'score': 0.05868387222290039,
  'token': 3688,
  'token_str': 'issues',
  'sequence': 'it seems important to tackle the climate issues.'},
 {'score': 0.047418754547834396,
  'token': 3406,
  'token_str': 'issue',
  'sequence': 'it seems important to tackle the climate issue.'},
 {'score': 0.027855267748236656,
  'token': 2629,
  'token_str': 'here',
  'sequence': 'it seems important to tackle the climate here.'}]